# ВКР

Интерактивный анализ; корень репозитория в `sys.path` — первая кодовая ячейка ниже.

In [1]:
# ─── Настройка путей ──────────────────────────────────────────────────────────
import sys, os
# Добавляем корень репозитория в путь
sys.path.insert(0, os.path.abspath(".."))

import logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")


## 1. Конфигурация

Измените пути и горизонты при необходимости.

In [2]:
from config import CFG

CFG["FILE_PATH"] = "../OZON_combined.csv"      # путь к CSV-файлу с котировками
CFG["HORIZONS"]  = [1, 5, 10]                  # горизонты прогнозирования (торг. дни)
CFG["OUT_DIR"]   = "../results"                # директория для артефактов

print("CFG загружен. Горизонты:", CFG["HORIZONS"])


CFG загружен. Горизонты: [1, 5, 10]


## 2. Smoke Test

Быстрая проверка загрузки и формата данных.

In [3]:
from main import smoke_test
smoke_test(CFG["FILE_PATH"])


2026-04-12 17:44:20,493 [INFO] Загружено: 1273 строк, 2020-11-24 — 2026-01-30
2026-04-12 17:44:20,494 [WARNING] [VALIDATE] Пропущено 3 рабочих дней (первые 5: [Timestamp('2020-12-31 00:00:00'), Timestamp('2021-01-01 00:00:00'), Timestamp('2021-01-07 00:00:00')])
2026-04-12 17:44:20,496 [INFO] [VALIDATE] Итог: 50 строк, 5 колонок, период 2020-11-24–2021-02-04
2026-04-12 17:44:20,501 [INFO] [INDICATORS] Признаки вычислены: 37 колонок
2026-04-12 17:44:20,501 [WARNING] [BUILDER] Отсутствуют признаки: {'sentiment_volatility', 'sentiment_trend_3', 'has_news', 'news_count', 'sentiment_score'}
2026-04-12 17:44:20,502 [INFO] [BUILDER] Удалено 33 строк с NaN (осталось 17)
2026-04-12 17:44:20,503 [INFO] [SMOKE] OK — df=(50, 37), X=(17, 29), reg_1=(49,)


## 3. Полный эксперимент

Цепочка как в `pipeline.run_experiment`: котировки → признаки → (опц.) новости → EDA → ARIMA–GARCH по `ARIMA_FORECAST_HORIZONS` → ML (walk-forward по `HORIZONS`).

`run_experiment` возвращает `data`, `arima_garch`, `ml`, `out_dir`. Имена файлов в `OUT_DIR` и смысл ключей — в `docs/MODEL_OVERVIEW.md`.

In [4]:
from pipeline import run_experiment

results = run_experiment(CFG, CFG["FILE_PATH"], CFG["OUT_DIR"])
ml = results["ml"]


2026-04-12 17:44:20,507 [INFO] [PIPELINE] Start experiment: data=../OZON_combined.csv, out=../results
2026-04-12 17:44:20,512 [INFO] Загружено: 1273 строк, 2020-11-24 — 2026-01-30
2026-04-12 17:44:20,517 [WARNING] [VALIDATE] Пропущено 97 рабочих дней (первые 5: [Timestamp('2020-12-31 00:00:00'), Timestamp('2021-01-01 00:00:00'), Timestamp('2021-01-07 00:00:00'), Timestamp('2021-02-23 00:00:00'), Timestamp('2021-03-08 00:00:00')])
2026-04-12 17:44:20,519 [WARNING] [VALIDATE] Выбросов (|z|>3.5): 41
2026-04-12 17:44:20,519 [WARNING] [VALIDATE] Даты: [Timestamp('2020-12-03 00:00:00'), Timestamp('2021-01-27 00:00:00'), Timestamp('2021-01-28 00:00:00'), Timestamp('2021-02-05 00:00:00'), Timestamp('2021-02-17 00:00:00'), Timestamp('2021-02-24 00:00:00'), Timestamp('2021-05-18 00:00:00'), Timestamp('2021-11-18 00:00:00'), Timestamp('2021-12-03 00:00:00'), Timestamp('2022-01-05 00:00:00')]
2026-04-12 17:44:20,519 [INFO] [VALIDATE] Итог: 1273 строк, 5 колонок, период 2020-11-24–2026-01-30
2026-0

## 4. Walk-Forward: регрессия

In [5]:
import pandas as pd

_reg_parts = []
for h, df in ml["wf_reg"].items():
    if df is not None and not df.empty:
        _reg_parts.append(df)
df_reg = pd.concat(_reg_parts, ignore_index=True) if _reg_parts else pd.DataFrame()

if not df_reg.empty:
    cols = [c for c in ["MAE", "RMSE", "MAPE", "MDA_%", "R2"] if c in df_reg.columns]
    display(df_reg.groupby(["model", "horizon"])[cols].mean().round(4))
else:
    print("Нет данных wf_reg.")


MAE    RMSE         MAPE   MDA_%      R2
model  horizon                                             
LGBreg 1        0.0167  0.0210  320883.4723  50.714 -0.4013
       5        0.0318  0.0377     421.1165  55.714 -1.4432
       10       0.0608  0.0706     435.3670  42.858 -8.9304
RFreg  1        0.0152  0.0195   31667.1422  46.427 -0.1345
       5        0.0293  0.0359     246.5442  46.428 -1.8460
       10       0.0598  0.0669     404.1087  39.285 -9.0741
Ridge  1        0.0153  0.0200  170960.1029  57.857 -0.1097
       5        0.0352  0.0421     359.3764  47.857 -2.8476
       10       0.0550  0.0636     394.8117  55.003 -6.4162

## 5. Walk-Forward: классификация

In [6]:
_clf_parts = []
for h, df in ml["wf_clf"].items():
    if df is not None and not df.empty:
        _clf_parts.append(df)
df_clf = pd.concat(_clf_parts, ignore_index=True) if _clf_parts else pd.DataFrame()

if not df_clf.empty:
    cols = [c for c in ["Accuracy", "F1", "AUC"] if c in df_clf.columns]
    display(df_clf.groupby(["model", "horizon"])[cols].mean().round(4))
else:
    print("Нет данных wf_clf.")


Accuracy      F1     AUC
model  horizon                          
LGBclf 1          0.5857  0.5005  0.5305
       5          0.4929  0.4876  0.6109
       10         0.4786  0.4945  0.5810
RFclf  1          0.5857  0.3794  0.5275
       5          0.4643  0.5010  0.4994
       10         0.4571  0.5260  0.4248

## 6. Бенчмарки: Buy & Hold и Naive

In [7]:
for h in CFG.get("HORIZONS", [1, 5, 10]):
    print(f"--- horizon h={h} ---")
    print("Naive(0):", ml["baseline"].get(h))
    print("Buy & Hold:", ml["buy_hold"].get(h))
    if ml["trading"].get(h):
        print("Trading (лучший clf):", ml["trading"][h])
    print("Stacking:", ml["stacking"].get(h))
    print()


--- horizon h=1 ---
Naive(0): {'model': 'Naive(0)', 'MAE': 0.020949, 'RMSE': 0.032348, 'MAPE': 99.2681, 'MDA_%': 0.73, 'R2': -0.0, 'IC': nan, 'IC_pval': nan}
Buy & Hold: {'model': 'Buy&Hold', 'Sharpe': 0.103, 'Calmar': 0.064, 'AnnualRet_%': 5.45, 'MaxDD_%': -84.51}
Trading (лучший clf): {'model': 'LGBclf_h1', 'Sharpe': 2.54, 'Sortino': 3.833, 'Calmar': 11.13, 'AnnualRet_%': 124.95, 'MaxDD_%': -11.23}
Stacking: {'model': 'Stacking(RF+LGB)', 'MAE': 0.01513, 'RMSE': 0.020846, 'MAPE': 37245.1642, 'MDA_%': 49.19, 'R2': -0.0165, 'IC': 0.0101, 'IC_pval': 0.8749}

--- horizon h=5 ---
Naive(0): {'model': 'Naive(0)', 'MAE': 0.049111, 'RMSE': 0.07149, 'MAPE': 99.6747, 'MDA_%': 0.0, 'R2': -0.0001, 'IC': nan, 'IC_pval': nan}
Buy & Hold: {'model': 'Buy&Hold', 'Sharpe': 0.177, 'Calmar': 0.223, 'AnnualRet_%': 22.27, 'MaxDD_%': -99.99}
Trading (лучший clf): {'model': 'LGBclf_h5', 'Sharpe': 0.013, 'Sortino': 0.02, 'Calmar': 0.019, 'AnnualRet_%': 0.67, 'MaxDD_%': -35.91}
Stacking: {'model': 'Stacking(RF+

## 7. Предсказания и диагностика остатков

In [8]:
_h = CFG["HORIZONS"][0] if CFG.get("HORIZONS") else 1
preds = ml["wf_preds"].get(_h)
if preds is not None and not preds.empty:
    display(preds.head(10))
    print(f"Предсказаний (h={_h}): {len(preds)} строк")
else:
    print("Предсказания недоступны.")


,y_true_reg,y_true_clf,pred_Ridge,pred_RFreg,pred_LGBreg,pred_RFclf,pred_LGBclf
DATE,,,,,,,
2025-01-20,0.032530,1,-0.006241,0.003041,0.015513,1,1
2025-01-21,-0.013353,0,-0.007192,0.003773,0.008509,1,0
2025-01-22,0.040518,1,-0.003913,0.004261,0.000707,1,1
2025-01-23,-0.014700,0,0.005075,0.005630,0.007911,1,1
2025-01-24,-0.023775,0,-0.005541,0.004706,0.006641,1,1
2025-01-27,0.023490,1,-0.008022,-0.003474,-0.003163,0,0
2025-01-28,-0.005284,0,-0.009260,0.000127,-0.001566,0,0
2025-01-29,0.035861,1,-0.012073,-0.004377,-0.006086,0,1
2025-01-30,-0.029581,0,-0.002828,0.002390,-0.010161,0,0


Предсказаний (h=1): 140 строк
